# Cart-pole control

In the second task you’ll try to craft controllers for the version of the cart-pole problem described by Barto, Sutton, and Anderson in [“Neuronlike Adaptive Elements That Can Solve Difficult Learning Control Problem”.](https://ieeexplore.ieee.org/document/6313077)

<div>
<img src="https://gymnasium.farama.org/_static/videos/mujoco/inverted_pendulum.gif" width="200"/>
</div>



# Gym env
This model has been encoded in the **Inverted Pendulum** environment

Find [here](https://gymnasium.farama.org/environments/mujoco/inverted_pendulum/) a detailed description of the environment
* action space
* observation space
* reward
* model

**The scope of the challenge is to find control laws / policies to set the force acting on the cart. This force must stabilize the pole upward**


In [ ]:
#@title Installing required libraries, setup

%%capture
!pip install gym pyvirtualdisplay
!apt-get install -y xvfb python-opengl ffmpeg
!pip install "gymnasium[mujoco]"
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# Env and benchmark policies
The following cell defines the environment, `InvertedPendulum-v5` and two baseline policie:
* `zero_policy`: always return 0, that is, no force is applied to the cart
* `random_policy`: a random force is sampled from the admissible intervals of the environment's action space

The effect of these two policies is shown in the following cells via animations of control scenarios.

In [2]:
import gymnasium as gym
import numpy as np
import random
import math
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create the InvertedPendulum environment (Mujoco must be installed)
env = gym.make("InvertedPendulum-v5", render_mode="rgb_array")

# Random policy - always chose [0]
def zero_policy(obs):
    return [0]

# Random policy
def random_policy(obs):
    return env.action_space.sample()

# Benchmark solutions
The following cells shows animations for the two benchmark policies. The animations halt when the policies are not able to respect the terminal condition (the absolute value of the vertical angle between the pole and the cart is greater than 0.2 radians).  
The cells below run 100 episodes for the `zero_policy` and the `random_policy` policies and retrieve the expected rewards in terms of number of steps before termination. Your solutions should beat both the benchmarks (~24 steps in expectation).

In [3]:
#@title Defining animating function
from gymnasium.wrappers import RecordVideo
import glob
from IPython.display import Video


def animate_policy(policy, env=gym.make("InvertedPendulum-v5", render_mode="rgb_array")):
  recorded = RecordVideo(
      env,                          # this is your AtariPreprocessing+FrameStack env
      video_folder="",
      episode_trigger=lambda ep: True,
      name_prefix="breakout_eval"
  )

  # Run exactly one episode
  obs, info = recorded.reset()
  done = False
  i=0
  obs, _ = env.reset()
  while not done:
      i+=1
      action = policy(obs)
      obs, reward, terminated, truncated, info = recorded.step(action)
      done = terminated or truncated
  recorded.close()

  # Find and embed the just-written MP4
  mp4 = sorted(glob.glob("breakout_eval-episode-*.mp4"))[-1]
  return Video(mp4, embed=True)

In [5]:
animate_policy(random_policy,env=gym.make("InvertedPendulum-v5", render_mode="rgb_array"))

In [6]:
animate_policy(zero_policy,env=gym.make("InvertedPendulum-v5", render_mode="rgb_array"))


In [11]:
def compute_average_reward(policy, env, num_episodes=100):
    total_reward = 0
    for _ in range(num_episodes):
        obs, _ = env.reset()
        done = False
        episode_reward = 0
        while not done:
            action = policy(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        total_reward += episode_reward
    return total_reward / num_episodes

# Create the environment
env = gym.make("InvertedPendulum-v5")

# Compute average reward for zero policy
avg_reward_zero = compute_average_reward(zero_policy, env)
print(f"Average reward for zero policy over 100 episodes: {avg_reward_zero}")

# Compute average reward for random policy
avg_reward_random = compute_average_reward(random_policy, env)
print(f"Average reward for random policy over 100 episodes: {avg_reward_random}")

env.close()

Average reward for zero policy over 100 episodes: 24.65
Average reward for random policy over 100 episodes: 5.43


# Point 1 - Find a reinforcement learning policy

<font color='red'>**Find a reinforcement learning policy stabilising the pole**</font>

For the RL policy you can use one of the following techniques:

- Tabular Q-learning
- Q-learning with function approximation
- DQN

In any case, keep a fixed maximum budget such that the notebook runs in a reasonable amount of time

---

## Approach: Q-learning with linear function approximation

### Why not tabular Q-learning?

The cart-pole state is a vector $(x, \theta, \dot{x}, \dot{\theta}) \in \mathbb{R}^4$. A tabular approach requires discretizing each dimension: with $d = 10$ bins per variable the table has $d^4 = 10{,}000$ state entries plus one row per discrete action. More importantly, the table does not generalize; learning that state $s$ has high value gives no information about a nearby state $s + \epsilon$. The relevant region of the state space (near the equilibrium $\theta \approx 0$) is only a small fraction of the full range, so most bins would never be visited.

### Linear function approximation

We parameterize the Q-function as:

$$\hat{q}(s, a;\, \theta_a) = \phi(s)^\top \theta_a$$

where $\phi: \mathbb{R}^4 \to \mathbb{R}^f$ is a **fixed** feature map and $\theta_a \in \mathbb{R}^f$ is a weight vector learned separately for each discrete action $a$. Similar states produce similar feature vectors, so the agent generalizes across the state space.

**Q-learning update.** Given a transition $(s_t, a_t, r_t, s_{t+1})$, the temporal-difference (TD) error is:

$$\delta_t = \underbrace{r_t + \gamma \max_{a'} \hat{q}(s_{t+1}, a';\, \theta_{a'})}_{\text{TD target}} - \hat{q}(s_t, a_t;\, \theta_{a_t})$$

Since $\hat{q}$ is linear, $\nabla_{\theta_{a_t}} \hat{q}(s_t, a_t) = \phi(s_t)$, so the gradient descent step simplifies to:

$$\theta_{a_t} \leftarrow \theta_{a_t} + \alpha\, \delta_t\, \phi(s_t)$$

This is model-free: we never use the physical equations of the pendulum; only observed transitions from the environment.

### Feature map: random Fourier features (RFF)

We construct $\phi$ using random Fourier features. For a bandwidth parameter $\sigma$, each feature is:

$$\phi_j(s) = \sqrt{\frac{2}{f}}\, \cos\!\bigl(\mathbf{w}_j^\top \tilde{s} + b_j\bigr), \qquad \mathbf{w}_j \sim \mathcal{N}\!\bigl(0,\, \sigma^{-2} I\bigr), \quad b_j \sim \mathrm{Uniform}(0, 2\pi)$$

where $\tilde{s} = s / s_{\text{scale}}$ is the state normalized by its expected operating range. The projections $\mathbf{w}_j$ and biases $b_j$ are sampled once at initialization and **never updated**; only $\theta$ is learned. The bandwidth $\sigma$ controls how quickly the features decay with distance between states.

### Action space discretization

The environment's action space is continuous: $F \in [-3, 3]$ N. Q-learning requires a discrete $\arg\max$ over actions, so we discretize into $k = 9$ evenly spaced values: $\{-3,\, -2.25,\, \ldots,\, 2.25,\, 3\}$. This resolution is sufficient for stabilization while keeping the $\arg\max$ cheap.

### Exploration

We use $\varepsilon$-greedy: with probability $\varepsilon$ the agent selects a random action (exploration), otherwise it selects $\arg\max_a \hat{q}(s, a)$ (exploitation). $\varepsilon$ decays multiplicatively from $1.0$ toward $0.01$ so that early episodes explore broadly and later ones refine the learned policy.


## Implementation and training

We train for 3,000 episodes. Each episode starts from a random initial state near the upright equilibrium (drawn from $\mathrm{Uniform}(-0.05, 0.05)$ per component) and terminates when $|\theta| \geq 0.2$ rad. After training, the greedy policy ($\varepsilon = 0$) is evaluated over 100 episodes and compared against the baselines established earlier (~24.65 steps for the zero policy).


In [ ]:
np.random.seed(3)

DISCRETE_ACTIONS: np.ndarray = np.linspace(-3.0, 3.0, 9)

# Normalization constants: each state component divided by its expected operating range
# so that inputs to the RFF encoder live roughly in [-1, 1].
STATE_SCALE: np.ndarray = np.array([2.0, 0.2, 5.0, 5.0])


class RFFEncoder:
    """Maps a continuous state vector to random Fourier features."""

    def __init__(self, state_dim: int, n_features: int, sigma: float) -> None:
        self._W: np.ndarray = np.random.normal(0.0, 1.0 / sigma, size=(n_features, state_dim))
        self._b: np.ndarray = np.random.uniform(0.0, 2.0 * np.pi, size=n_features)
        self._scale: float = float(np.sqrt(2.0 / n_features))

    def encode(self, state: np.ndarray) -> np.ndarray:
        return self._scale * np.cos(self._W @ state + self._b)


class LinearQAgent:
    """Q-learning agent with one linear weight vector per discrete action."""

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float,
        gamma: float,
        epsilon_start: float,
        epsilon_min: float,
        epsilon_decay: float,
    ) -> None:
        self._n_actions = n_actions
        self._alpha = alpha
        self._gamma = gamma
        self._epsilon = epsilon_start
        self._epsilon_min = epsilon_min
        self._epsilon_decay = epsilon_decay
        # One weight vector per action, zero-initialized
        self._weights: np.ndarray = np.zeros((n_actions, n_features))

    @property
    def epsilon(self) -> float:
        return self._epsilon

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        return self._weights @ phi

    def select_action(self, phi: np.ndarray) -> int:
        """Epsilon-greedy: random with probability epsilon, greedy otherwise."""
        if np.random.rand() < self._epsilon:
            return int(np.random.randint(self._n_actions))
        return int(np.argmax(self._q_values(phi)))

    def act(self, phi: np.ndarray) -> int:
        """Pure greedy action selection, used at evaluation time."""
        return int(np.argmax(self._q_values(phi)))

    def update(
        self, phi: np.ndarray, action: int, reward: float,
        phi_next: np.ndarray, done: bool
    ) -> None:
        """Single Q-learning gradient step on the TD error."""
        q_current = float(self._weights[action] @ phi)
        q_next = 0.0 if done else float(np.max(self._q_values(phi_next)))
        td_error = reward + self._gamma * q_next - q_current
        self._weights[action] += self._alpha * td_error * phi
        # Multiplicative epsilon decay applied once per step
        self._epsilon = max(self._epsilon_min, self._epsilon * self._epsilon_decay)

In [ ]:
# Hyperparameters
N_FEATURES = 400
SIGMA      = 0.5    # RBF bandwidth relative to normalized state scale
ALPHA      = 0.005  # Learning rate
GAMMA      = 0.99   # Discount factor
EPS_START  = 1.0
EPS_MIN    = 0.01
EPS_DECAY  = 0.999  # Per-step decay; reaches EPS_MIN after ~4600 steps
N_EPISODES = 3000

encoder = RFFEncoder(state_dim=4, n_features=N_FEATURES, sigma=SIGMA)
agent = LinearQAgent(
    n_features=N_FEATURES,
    n_actions=len(DISCRETE_ACTIONS),
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon_start=EPS_START,
    epsilon_min=EPS_MIN,
    epsilon_decay=EPS_DECAY,
)

env_train = gym.make("InvertedPendulum-v5")
episode_rewards: list[float] = []

for _ in range(N_EPISODES):
    obs, _ = env_train.reset()
    phi = encoder.encode(obs / STATE_SCALE)
    done = False
    total_r = 0.0
    while not done:
        a_idx = agent.select_action(phi)
        obs_next, reward, terminated, truncated, _ = env_train.step([DISCRETE_ACTIONS[a_idx]])
        done = terminated or truncated
        phi_next = encoder.encode(obs_next / STATE_SCALE)
        agent.update(phi, a_idx, reward, phi_next, terminated)
        phi = phi_next
        total_r += reward
    episode_rewards.append(total_r)

env_train.close()
print(f"Training complete. Final epsilon: {agent.epsilon:.4f}")

# Evaluation: greedy policy over 100 episodes
def rl_policy(obs: np.ndarray) -> list[float]:
    phi = encoder.encode(obs / STATE_SCALE)
    return [float(DISCRETE_ACTIONS[agent.act(phi)])]

env_eval = gym.make("InvertedPendulum-v5")
avg_reward_rl = compute_average_reward(rl_policy, env_eval)
env_eval.close()

print(f"\nAverage reward  RL policy:   {avg_reward_rl:.1f} steps")
print(f"Average reward  zero policy:   {avg_reward_zero:.1f} steps  (benchmark)")
print(f"Average reward  random policy: {avg_reward_random:.1f} steps  (benchmark)")

# Learning curve
window = 50
smoothed = np.convolve(episode_rewards, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(smoothed, label=f"{window}-episode moving average")
plt.axhline(avg_reward_zero, color="orange", linestyle="--", label=f"zero policy ({avg_reward_zero:.0f} steps)")
plt.xlabel("Episode")
plt.ylabel("Total reward (steps survived)")
plt.title("Q-learning with Linear Function Approximation - Training curve")
plt.legend()
plt.tight_layout()
plt.show()


## Discussion

The trained policy should clearly outperform both baselines. A few observations about design choices and their effect:

**Episode length ceiling.** The environment truncates episodes at 1,000 steps. The zero-policy baseline achieves ~24 steps, so any policy that consistently stabilizes the pole approaches 1,000. A well-trained agent should reach several hundred steps on average; hitting the ceiling on most evaluation episodes would indicate the policy has converged.

**Bandwidth $\sigma = 0.5$.** In normalized state space, $\sigma = 0.5$ means features decay to near-zero for states roughly $0.5 \times s_{\text{scale}}$ apart. This matches the relevant perturbation range: episodes terminate at $|\theta| = 0.2$ rad, which corresponds to $\tilde{\theta} = 1.0$ in normalized coordinates. A smaller $\sigma$ would make features too local and slow learning; a larger one would blur value differences across states that the agent needs to distinguish.

**State normalization.** The environment observation space is unbounded ($\text{Box}(-\infty, +\infty)$ for all components). Dividing by STATE\_SCALE ensures inputs to the RFF encoder stay near $[-1, 1]$ in the operating region, which is necessary for the RBF bandwidth $\sigma$ to be meaningful.

**Epsilon decay rate.** With $\varepsilon_{\text{decay}} = 0.999$ per step and episodes averaging roughly 100 steps during the early learning phase, epsilon reaches $0.01$ after about 4,600 environment steps. This is well within the 3,000-episode budget once later episodes survive longer.

**Action discretization.** Nine uniform values over $[-3, 3]$ give a resolution of $0.75$ N. For a stabilization task this is sufficient; finer discretization would increase the $\arg\max$ cost without meaningfully improving the policy.

**Limitation vs. model-based approaches.** Q-learning here is model-free: it never uses the linearized equations of the pendulum. The convergence guarantee of tabular Q-learning does not extend directly to linear function approximation with off-policy updates. In practice it converges because the problem is low-dimensional and the RFF features cover the relevant state region well. For Tasks 2 and 3 we switch to LQR and MPC, which exploit the known dynamics directly and come with stronger stability guarantees.

---

# Point 2 - Find a policy using LQR/MPC/DP
<font color='red'>**Find a policy belonging to the following classes stabilising the pole**</font>

- LQR with dynamic programming
- MPC

For this task you’ll rely on the linearized equations for this system. Since we are interested in controlling the system close to its equilibrium point (vertical pole), the dynamic equations provided can be linearized around $\theta = 0$.

# Point 3 - Find a policy under uncertainty

For the last task, you will use a different environment, in which a disturbance is added to your policy action:

$a = a + \epsilon$

where $\epsilon \sim \mathcal{N}(\mu = 0, \sigma = 0.25)$

The wrapper below injects an additive disturbance into the action sent to the MuJoCo environment, then clips the result to the original action bounds. This lets us test whether a policy remains stable under actuator noise without changing the policy itself.

<font color='red'>**Either adapt one of the policies you used before or find a new one rejecting the noise.**</font>

In [ ]:
class ActionDisturbanceWrapper(gym.ActionWrapper):
    def __init__(self, env, disturbance_std=0.25, disturbance_bias=0.0, seed=None):
        super().__init__(env)
        self.disturbance_std = disturbance_std
        self.disturbance_bias = disturbance_bias
        self.rng = np.random.default_rng(seed)

    def action(self, action):
        action = np.asarray(action, dtype=np.float32)
        disturbance = self.rng.normal(
            loc=self.disturbance_bias,
            scale=self.disturbance_std,
            size=action.shape,
        )
        disturbed_action = action + disturbance
        return np.clip(disturbed_action, self.action_space.low, self.action_space.high).astype(np.float32)


def make_disturbed_env(render_mode=None, disturbance_std=0.25, disturbance_bias=0.0, seed=None):
    env = gym.make("InvertedPendulum-v5", render_mode=render_mode)
    return ActionDisturbanceWrapper(
        env,
        disturbance_std=disturbance_std,
        disturbance_bias=disturbance_bias,
        seed=seed,
    )